# 2️⃣ Circuit and Feature Analysis

> ##### Learning Objectives
>
> * Apply your understanding of the 1D and 2D Fourier bases to show that the activations / effective weights of your model are highly sparse in the Fourier basis.
> * Turn these observations into concrete hypotheses about the model's algorithm.
> * Verify these hypotheses using statistical methods, and interventions like ablation.
> * Fully understand the model's algorithm, and how it solves the task.

Understanding a transformer breaks down into two high-level parts - interpreting the features represented by non-linear activations (output probabilities, attention patterns, neuron activations), and interpreting the circuits that calculate each feature (the weights representing the computation done to convert earlier features into later features). In this section we interpret the embedding, the neuron activations (a feature) and the logit computation (a circuit). These are the most important bits to interpet.

Let's start with the embedding.

## Understanding the embedding

Below is some code to plot the embedding in the Fourier basis. You should run this code, and interpret the output.

In [ ]:
utils.line(
    (fourier_basis @ W_E).pow(2).sum(1),
    hover=fourier_basis_names,
    title="Norm of embedding of each Fourier Component",
    xaxis="Fourier Component",
    yaxis="Norm",
)

<details>
<summary>Interpretation</summary>

You should find that the embedding is sparse in the Fourier basis, and throws away all Fourier components apart from a handful of frequencies (the number of frequencies and their values are arbitrary, and vary between training runs).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/fourier_norms.png" width="500">

The Fourier basis vector for component $2k$ is:

$$
\cos\left(\frac{2 \pi k}{p}x\right)_{x = 0, ..., p-1}
$$

and same for $2k-1$, but with $\cos$ replaced with $\sin$.

So this result tells us that, for the input `x`, we're keeping the information $\cos\left(\frac{2 \pi k}{p}x\right)$ for each of the key frequencies $k$, and throwing away this information for all other frequencies $\omega$.

Let us term the frequencies with non-trivial norm the **key frequencies** (here, 14, 31, 35, 41, 42, 52).

</details>

<details>
<summary>Another perspective (from singular value decomposition)</summary>

Recall that we can write any matrix as the product of an orthogonal matrix, a diagonal matrix, and another orthogonal matrix:

$$
\begin{aligned}
A &= U S V^T \\
    &= \sum_{i=1}^k \sigma_i u_i v_i^T
\end{aligned}
$$

where $u_i$, $v_i$ are the column vectors for the orthogonal matrices $U$, $V$, and $\sigma_1, ..., \sigma_k$ are the non-zero singular values of $A$. If this isn't familiar, you might want to go through the induction heads exercises, which discusses SVD in more detail. This is often a natural way to represent low-rank matrices (because most singular values $\sigma_i$ will be zero).

Denote the matrix `fourier_basis` as $F$ (remember that the rows are basis vectors). Here, we've found that the matrix $F W_E$ is very sparse (i.e. most of its row vectors are zero). From this, we can deduce that $W_E$ is well-approximated by the following low-rank SVD:

$$
W_E \approx F^T S V^T
$$

because then left-multiplying by $F$ gives us a matrix with zeros everywhere except for the rows corresponding to non-zero singular values:

$$
F W_E \approx F^T F S V^T = S V^T
$$

In other words, the **input directions** we're projecting onto when we take the embedding of our tokens $t_0$, $t_1$ are approximately the Fourier basis directions.

To directly visualise the matrix product $S V^T$ (which will be sparse in rows because most diagonal values of $S$ are zero), run the following:

```python
imshow_div(fourier_basis @ W_E)
```
</details>

## Understanding neuron activations

Now we've established that $W_E$ is only preserving the information corresponding to a few key frequencies, let's look at what those frequencies are actually being used for.

> TL;DR:
> * Each neuron's activations are a linear combination of the constant term and the **linear and quadratic terms of a specific frequency**.
> * The neurons clearly **cluster according to the key frequencies**.

### Neurons produce quadratic terms

First, we recall the diagrams of the neuron activations in the input basis and the 2D Fourier basis, which we made in the previous section:

```python
top_k = 5
utils.inputs_heatmap(neuron_acts_post_sq[..., :top_k], ...)
utils.imshow_fourier(neuron_acts_post_fourier_basis[..., :top_k], ...)
```

We found that the first plot looked periodic (recall, periodic in standard basis = sparse in Fourier basis), and the second gave us more details about the Fourier basis representation by showing that **each neuron was associated with some key frequency**, from one of the key frequencies we observered earlier.

For instance, look at the first neuron in these plots. We can see that the only frequencies which matter in the 2D Fourier basis are the constant terms and the frequencies corresponding to $\omega = 42$ (we get both $\sin$ and $\cos$ terms). In total, this gives us nine terms, which are (up to scale factors):

$$
\begin{bmatrix}
1 & \cos(\omega_k x) & \sin(\omega_k x) \\
\cos(\omega_k y) & \cos(\omega_k x)\cos(\omega_k y) & \sin(\omega_k x)\cos(\omega_k y) \\
\sin(\omega_k y) & \cos(\omega_k x)\sin(\omega_k y) & \sin(\omega_k x)\sin(\omega_k y)
\end{bmatrix}
$$

where $\omega_k = 2 \pi k / p$, and in this case $k = 42$. These include the constant term, four linear terms, and four quadratic terms.

What is the significance of this? Importantly, we have the following trig formulas:

$$
\begin{aligned}
\cos(\omega_k(x + y)) = \cos(\omega_k x) \cos(\omega_k y) - \sin(\omega_k x) \sin(\omega_k y) \\
\sin(\omega_k(x + y)) = \sin(\omega_k x) \cos(\omega_k y) + \cos(\omega_k x) \sin(\omega_k y)
\end{aligned}
$$

The plots tell us that some of the terms on the right of these equations (i.e. the quadratic terms) are the ones being detected by our neurons. Since we know that the model will eventually have to internally represent the quantity $x+y$ in some way in order to perform modular addition, we might guess that this is how it does it (i.e. it calculates the quantities on the left hand side of the equations, by first computing the things on the right). In other words, our neurons are in some sense storing the information $\cos(\omega_k(x + y))$ and $\sin(\omega_k(x + y))$, for different frequencies $k$.

Let's have a closer look at some of the coefficients for these 2D Fourier basis terms.

### Exercise - calculate the mean squared coefficient

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> This exercise asks you to perform some basic operations and make some simple plots with your neuron activations.
> ```

We might speculate that the neuron activation (which is a function of the inputs $x$, $y$) is in some sense trying to approximate the function $\sum_i \sum_j B_{i, j} F_{i, x} F_{j, y}$, where:

* $F$ is the Fourier change-of-basis matrix.
    * Note that, for example, $F_{1,17}= \cos(\frac{2\pi}{p} 17)$, and $F_{2, 17} = \sin(\frac{2\pi}{p} 17)$.
* $B$ is a matrix of coefficients, which we suspect is highly sparse.
    * Specifically, we suspect that the "true function" our model is trying to approximate looks something like a linear combination of the nine terms in the matrix above:

$$
\sum_{i\in \{0, 2\omega-1, 2\omega\}} \sum_{j\in \{0, 2\omega-1, 2\omega\}} B_{i,j} F_{i,x} F_{j,y}
$$

(recall that the $2\omega-1$-th and $2\omega$-th basis vectors are the cosine and sine vectors for frequency $\omega$).

Create a heatmap of the mean squared coefficient for each neuron (in other words, the `(i, j)`-th value in your heatmap is the mean of $B_{i, j}^2$ across all neurons).

Your code should involve two steps:

* Centering the neuron activations, by subtracting the mean over all batches (because this essentially removes the bias term). All ReLU neurons always have non-negative activations, and the constant term should usually be considered separately.
* Taking the 2D Fourier transform of the centered neuron activations.

We've given you the code to plot your results.

Remember to work with the `neuron_acts_post_sq` object, which already has its batch dimensions shaped into a grid.

In [ ]:
# YOUR CODE HERE - compute neuron activations (centered)
neuron_acts_centered = neuron_acts_post_sq - neuron_acts_post_sq.mean((0, 1), keepdim=True)

# Take 2D Fourier transform
neuron_acts_centered_fourier = fft2d(neuron_acts_centered)


utils.imshow_fourier(
    neuron_acts_centered_fourier.pow(2).mean(-1),
    title="Norms of 2D Fourier components of centered neuron activations",
)

Your plot of the average $B_{i, j}^2$ values should show the kind of sparsity you saw earlier: the only non-zero terms will be the const, linear and quadratic terms corresponding to a few select frequencies. You can compare these frequencies to the **key frequencies** we defined earlier, and validate that they're the same.

### How do we get the quadratic terms?

Exactly how these quadratic terms are calculated is a bit convoluted. A good mental model for neural networks is that they are really good at matrix multiplication and addition, and anything else takes a lot of effort. So we should expect taking the product of two different parts of the input to be pretty hard!

You can see the [original notebook](https://colab.research.google.com/drive/1F6_1_cWXE5M7WocUcpQWp3v8z4b1jL20#scrollTo=ByvnsimvZXGC) for some more details on this calculation. The short version - **the model uses both ReLU activations and element-wise products with attention to multiply terms, in hacky ways**.

The attention pattern products (which we won't discuss much here) work because attention involves taking the product of value vectors and attention probabilities, which are each functions of the input. This seems a bit weird because attention probabilities and value vectors are usually thought of as playing different roles, but here they're working together to allow the model to multiply two different parts of the input.

The ReLU activations also pretty surprising. It turns out that linear functions of 1D and 2D Fourier components are well approximated by the ReLU of a linear function of the 1D Fourier components. Specifically, if we approximate the expression:

$$
\operatorname{ReLU}(A + B \cos(\omega x) + B \cos(\omega y))
$$

(for $A$, $B > 0$) as a linear combination of the following 4 terms in the 2D Fourier basis:

$$
\alpha + \beta \cos(\omega x) + \beta \cos(\omega y) + \gamma \cos(\omega x) \cos(\omega y)
$$

we find that it includes a significant component in the $\cos(\omega x) \cos(\omega y)$ direction. The key intuition for why this happens is that the quadratic term captures **interaction between $x$ and $y$**. $\operatorname{ReLU}$ is a [convex function](https://en.wikipedia.org/wiki/Convex_function) of $\cos(\omega x) + \cos(\omega y)$, so (if we pretend $x$ and $y$ are random variables) it is larger in expectation when these two inputs are correlated, hence $\gamma > 0$. \*

\* *Note - this is quite a handwavey argument, so don't worry too much if it doesn't seem intuitive!*

This is important because our model can calculate the first of these two expressions (it can take linear combinations of $\cos(\omega x)$ and $\cos(\omega y)$ in the attention layer, then apply $\operatorname{ReLU}$ during the MLP), but it can't directly calculate the second expression. So we can essentially use our ReLU to approximate a sum of linear and quadratic terms.

### Exercise - verify the quadratic term matters, and that $\gamma > 0$

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> Doing this exercise isn't super valuable to the overall experience of this section. You can skip it if you want.
> ```

Take $A = {1}/{2\sqrt{p}}, \; B = 1, \;$ and $\;\omega = \omega_{42} = (2 \pi \times 42) / p \;$ (one of the key frequencies we observed earlier). Find the coefficients $\alpha$, $\beta$ and $\gamma$ that minimize the mean squared error between the ReLU approximation and the true quadratic function. Find the $r^2$ score of this fit. Verify that $\gamma > 0$, and the score is close to 1. Also, verify that the $r^2$ score decreases by quite a lot when you omit the quadratic term (showing that this quadratic term is important).

You can use the [`LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) function from `sklearn`.

*Remember to use the normalized 1D and 2D Fourier basis vectors in your regression.*

In [ ]:
from sklearn.linear_model import LinearRegression

# Choose a particular frequency, and get the corresponding cosine basis vector
k = 42
idx = 2 * k - 1
vec = fourier_basis[idx]

# Get ReLU function values
relu_func_values = F.relu(0.5 * (p**-0.5) + vec[None, :] + vec[:, None])

# Get terms we'll be using to approximate it
# Note we're including the constant term here
data = t.stack(
    [fourier_2d_basis_term(i, j) for (i, j) in [(0, 0), (idx, 0), (0, idx), (idx, idx)]], dim=-1
)

# Reshape, and convert to numpy
data = to_numpy(data.reshape(p * p, 4))
relu_func_values = to_numpy(relu_func_values.flatten())

# Fit a linear model (we don't need intercept because we have const Fourier basis term)
reg = LinearRegression(fit_intercept=False).fit(data, relu_func_values)
coefs = reg.coef_
r2 = reg.score(data, relu_func_values)
print(
    "ReLU(0.5 + cos(wx) + cos(wy)) ≈ {:.3f}*const + {:.3f}*cos(wx) + {:.3f}*cos(wy) + {:.3f}*cos(wx)cos(wy)".format(
        *coefs
    )
)
print(f"r2: {r2:.3f}")

# Run the regression again, but without the quadratic term
data = data[:, :3]
reg = LinearRegression().fit(data, relu_func_values)
coefs = reg.coef_
bias = reg.intercept_
r2 = reg.score(data, relu_func_values)
print(f"r2 (no quadratic term): {r2:.3f}")

<details>
<summary>Discussion of results</summary>

You should get the following:

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">ReLU(0.5 + cos(wx) + cos(wy)) ≈ 9.190*const + 6.807*cos(wx) + 6.807*cos(wy) + 3.566*cos(wx)cos(wy)
r2: 0.966
r2 (no quadratic term): 0.849</pre>

This confirms that the quadratic term does indeed have a positive coefficient, and that it explains a lot of the variance in the ReLU function (specifically, it explains over 2/3 of the variance which is left after we've accounted for the linear terms).

(Note that we didn't print out the coefficients for the regression without quadratic terms - our 2D Fourier basis vectors are orthogonal, so we can guarantee that these coefficients would be the same as the coefficients in the regression with quadratic terms.)

</details>

### Neurons cluster by frequency

Now that we've established that the neurons each seem to have some single frequency that they're most sensitive to (and ignore all others), let's try and sort the neurons by this frequency, and see how effective each of these frequencies are at explaining the neurons' behaviour.

### Exercise - find neuron clusters

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-30 minutes on this exercise.
> This exercise is conceptually important, but quite challenging.
> ```

For each neuron, you should find the frequency such that the Fourier components containing that frequency explain the largest amount of the variance of that neuron's activation (in other words, which frequency is such that the sum of squares of the const, linear and quadratic terms of that frequency for this particular is largest, as a fraction of the sum of squares of all the Fourier coefficients for this neuron).

We've provided you with the helper function `arrange_by_2d_freqs`. This takes in a tensor of coefficients in the 2D Fourier basis (with shape `(p, p, ...)`), and returns a tensor of shape `(p//2, 3, 3, ...)`, representing the Fourier coefficients sorted into each frequency. In other words, the `[k-1, ...]`-th slice of this tensor will be the `(3, 3, ...)`-shape tensor containing the Fourier coefficients for the following (normalized) 2D Fourier basis vectors:

$$
\begin{bmatrix}
1 & \cos(\omega_k x) & \sin(\omega_k x) \\
\cos(\omega_k y) & \cos(\omega_k x)\cos(\omega_k y) & \sin(\omega_k x)\cos(\omega_k y) \\
\sin(\omega_k y) & \cos(\omega_k x)\sin(\omega_k y) & \sin(\omega_k x)\sin(\omega_k y)
\end{bmatrix}
$$

Think of this as just a fancy rearranging of the tensor, so that you're more easily able to access all the (const, linear, quadratic) terms for any particular frequency.

In [ ]:
def arrange_by_2d_freqs(tensor: Tensor) -> Tensor:
    """
    Takes a tensor of shape (p, p, *batch_dims), returns tensor of shape (p//2 - 1, 3, 3, *batch_dims)
    representing the Fourier coefficients sorted by frequency (each slice contains const, linear and
    quadratic terms).

    In other words, if `tensor` had shape (p, p) and looked like this:

        1           cos(w_1*x)            sin(w_1*x)           ...
        cos(w_1*y)  cos(w_1*x)cos(w_1*y)  sin(w_1*x)cos(w_1*y) ...
        sin(w_1*y)  cos(w_1*x)sin(w_1*y)  sin(w_1*x)sin(w_1*y) ...
        cos(w_2*y)  cos(w_1*x)cos(w_2*y)  sin(w_1*x)cos(w_2*y) ...
        ...         ...                   ...

    Then arrange_by_2d_freqs(tensor)[k-1] should be the following (3, 3) tensor of kth-mode Fourier
    frequencies:

        1           cos(w_k*x)            sin(w_k*x)
        cos(w_k*y)  cos(w_k*x)cos(w_k*y)  sin(w_k*x)cos(w_k*y)
        sin(w_k*y)  cos(w_k*x)sin(w_k*y)  sin(w_k*x)sin(w_k*y)

    for k = 1, 2, ..., p//2. Note we omit the constant term, i.e. the 0th slice has frequency k=1.

    Any dimensions beyond the first 2 are treated as batch dimensions, i.e. we only rearrange the
    first 2.
    """
    idx_2d_y_all = []
    idx_2d_x_all = []
    for freq in range(1, p // 2):
        idx_1d = [0, 2 * freq - 1, 2 * freq]
        idx_2d_x_all.append([idx_1d for _ in range(3)])
        idx_2d_y_all.append([[i] * 3 for i in idx_1d])
    return tensor[idx_2d_y_all, idx_2d_x_all]


def find_neuron_freqs(
    fourier_neuron_acts: Float[Tensor, "p p d_mlp"],
) -> tuple[Float[Tensor, "d_mlp"], Float[Tensor, "d_mlp"]]:
    """
    Returns the tensors `neuron_freqs` and `neuron_frac_explained`, containing the frequencies that
    explain the most variance of each neuron and the fraction of variance explained, respectively.
    """
    fourier_neuron_acts_by_freq = arrange_by_2d_freqs(fourier_neuron_acts)
    assert fourier_neuron_acts_by_freq.shape == (p // 2 - 1, 3, 3, utils.d_mlp)

    # Sum squares of all frequency coeffs, for each neuron
    square_of_all_terms = einops.reduce(
        fourier_neuron_acts.pow(2), "x_coeff y_coeff neuron -> neuron", "sum"
    )

    # Sum squares just corresponding to const+linear+quadratic terms,
    # for each frequency, for each neuron
    square_of_each_freq = einops.reduce(
        fourier_neuron_acts_by_freq.pow(2), "freq x_coeff y_coeff neuron -> freq neuron", "sum"
    )

    # Find the freq explaining most variance for each neuron
    # (and the fraction of variance explained)
    neuron_variance_explained, neuron_freqs = square_of_each_freq.max(0)
    neuron_frac_explained = neuron_variance_explained / square_of_all_terms

    # The actual frequencies count up from k=1, not 0!
    neuron_freqs += 1

    return neuron_freqs, neuron_frac_explained


neuron_freqs, neuron_frac_explained = find_neuron_freqs(neuron_acts_centered_fourier)
key_freqs, neuron_freq_counts = t.unique(neuron_freqs, return_counts=True)

assert key_freqs.tolist() == [14, 35, 41, 42, 52]

print("All tests for `find_neuron_freqs` passed!")

Once you've written this function and passed the tests, you can plot the fraction of variance explained.

In [ ]:
fraction_of_activations_positive_at_posn2 = (cache["pre", 0][:, -1] > 0).float().mean(0)

utils.scatter(
    x=neuron_freqs,
    y=neuron_frac_explained,
    xaxis="Neuron frequency",
    yaxis="Frac explained",
    colorbar_title="Frac positive",
    title="Fraction of neuron activations explained by key freq",
    color=to_numpy(fraction_of_activations_positive_at_posn2),
)

We color the neurons according to the fraction of data points for which they are active. We see that there are 5 distinct clusters of neurons that are well explained (frac > 0.85) by one frequency.

There is a sixth, diffuse cluster of neurons that always fire. They are not well-explained by any particular frequency. This makes sense, because since ReLU acts as an identity on this cluster, there's no reason to privilege the neuron basis (i.e. there's no reason to expect that the specific value of this neuron's activations has any particular meaning in relation to the Fourier components of the input, since we could just as easily apply rotations to the always-firing neurons).

In [ ]:
# To represent that they are in a special sixth cluster, we set the frequency of these neurons to -1
neuron_freqs[neuron_frac_explained < 0.85] = -1.0
key_freqs_plus = t.concatenate([key_freqs, -key_freqs.new_ones((1,))])

for i, k in enumerate(key_freqs_plus):
    print(f"Cluster {i}: freq k={k}, {(neuron_freqs == k).sum()} neurons")

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">Cluster 0: freq k=14, 44 neurons
Cluster 1: freq k=35, 93 neurons
Cluster 2: freq k=41, 145 neurons
Cluster 3: freq k=42, 87 neurons
Cluster 4: freq k=52, 64 neurons
Cluster 5: freq k=-1, 79 neurons</pre>

### Further investigation of neuron clusters

We can separately view the norms of the Fourier Components of the neuron activations for each cluster. The following code should do the same thing as your plot of the average $B_{i, j}^2$ values earlier, except it sorts the neurons into clusters by their frequency before taking the mean.

*(Note, we're using the argument `facet_col` rather than `animation_frame`, so we can see all the plots at once.)*

In [ ]:
fourier_norms_in_each_cluster = []
for freq in key_freqs:
    fourier_norms_in_each_cluster.append(
        einops.reduce(
            neuron_acts_centered_fourier.pow(2)[..., neuron_freqs == freq],
            "batch_y batch_x neuron -> batch_y batch_x",
            "mean",
        )
    )

utils.imshow_fourier(
    t.stack(fourier_norms_in_each_cluster),
    title="Norm of 2D Fourier components of neuron activations in each cluster",
    facet_col=0,
    facet_labels=[f"Freq={freq}" for freq in key_freqs],
)

Now that we've found what appear to be neuron clusters, it's time to validate our observations. We'll do this by showing that, for each neuron cluster, we can set terms for any other frequency to zero and still get good performance on the task.

### Exercise - validate neuron clusters

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 25-40 minutes on the following exercises.
> They are designed to get you to engage with ideas from linear algebra (specifically projections), and are very important conceptually.
> ```

We want to do the following:

* Take `neuron_acts_post`, which is a tensor of shape `(p*p, d_mlp)`, with the `[i, j]`-th element being the activation of neuron `j` on the `i`-th input sequence in our `all_data` batch.
* Treating this tensor as `p` separate vectors in $\mathbb{R}^{p^2}$ (with the last dimension being the batch dimension), we will project each of these vectors onto the subspace of $\mathbb{R}^{p^2}$ spanned by the 2D Fourier basis vectors for the associated frequency of that particular neuron (i.e. the constant, linear, and quadratic terms).
* Take these projected `neuron_acts_post` vectors, and apply `W_logit` to give us new logits. Compare the cross entropy loss with these logits to our original logits.

If our hypothesis is correct (i.e. that for each cluster of neurons associated with a particular frequency, that frequency is the only one that matters), then our loss shouldn't decrease by much when we project out the other frequencies in this way.

First, you'll need to write the function `project_onto_direction`. This takes two inputs: `batch_vecs` (a batch of vectors, with batch dimension at the end) and `v` (a single vector), and returns the projection of each vector in `batch_vecs` onto the direction `v`.

In [ ]:
def project_onto_direction(batch_vecs: Tensor, v: Tensor) -> Tensor:
    """
    Returns the component of each vector in `batch_vecs` in the direction of `v`.

    batch_vecs.shape = (n, ...)
    v.shape = (n,)
    """
    # Get tensor of components of each vector in v-direction
    components_in_v_dir = einops.einsum(batch_vecs, v, "n ..., n -> ...")

    # Use these components as coefficients of v in our projections
    return einops.einsum(components_in_v_dir, v, "..., n -> n ...")


tests.test_project_onto_direction(project_onto_direction)

<details>
<summary>Hint</summary>

Recall that the projection of $w$ onto $v$ (for a normalized vector $v$) is given by:

$$
w_{proj} = (w \cdot v) v
$$

You might find it easier to do this in two steps: first calculate the components in the $v$-direction by taking the inner product along the 0th dimension, then create a batch of multiples of $v$, scaled by these components.
</details>

Next, you should write the function `project_onto_frequency`. This takes a batch of vectors with shape `(p**2, batch)`, and a frequency `freq`, and returns the projection of each vector onto the subspace spanned by the nine 2D Fourier basis vectors for that frequency (i.e. one constant term, four linear terms, and four quadratic terms).

In [ ]:
def project_onto_frequency(batch_vecs: Tensor, freq: int) -> Tensor:
    """
    Returns the projection of each vector in `batch_vecs` onto the
    2D Fourier basis directions corresponding to frequency `freq`.

    batch_vecs.shape = (p**2, ...)
    """
    assert batch_vecs.shape[0] == p**2

    return sum(
        [
            project_onto_direction(
                batch_vecs,
                fourier_2d_basis_term(i, j).flatten(),
            )
            for i in [0, 2 * freq - 1, 2 * freq]
            for j in [0, 2 * freq - 1, 2 * freq]
        ]
    )


tests.test_project_onto_frequency(project_onto_frequency)

Finally, run the following code to project out the other frequencies from the neuron activations, and compare the new loss. You should make sure you understand what this code is doing.

In [ ]:
logits_in_freqs = []

for freq in key_freqs:
    # Get all neuron activations corresponding to this frequency
    filtered_neuron_acts = neuron_acts_post[:, neuron_freqs == freq]

    # Project onto const/linear/quadratic terms in 2D Fourier basis
    filtered_neuron_acts_in_freq = project_onto_frequency(filtered_neuron_acts, freq)

    # Calcluate new logits, from these filtered neuron activations
    logits_in_freq = filtered_neuron_acts_in_freq @ W_logit[neuron_freqs == freq]

    logits_in_freqs.append(logits_in_freq)

# We add on neurons in the always firing cluster, unfiltered
logits_always_firing = neuron_acts_post[:, neuron_freqs == -1] @ W_logit[neuron_freqs == -1]
logits_in_freqs.append(logits_always_firing)

# Compute new losses
key_freq_loss = utils.test_logits(
    sum(logits_in_freqs), bias_correction=True, original_logits=original_logits
)
key_freq_loss_no_always_firing = utils.test_logits(
    sum(logits_in_freqs[:-1]), bias_correction=True, original_logits=original_logits
)

# Print new losses
print(f"""Loss with neuron activations ONLY in key freq (including always firing cluster): {key_freq_loss:.3e}
Loss with neuron activations ONLY in key freq (exclusing always firing cluster): {key_freq_loss_no_always_firing:.3e}
Original loss: {original_loss:.3e}
""")

You should find that the loss doesn't change much when you project out the other frequencies, and even if you remove the always firing cluster the loss is still very small.

We can also compare the importance of each cluster of neurons by ablating it (while continuing to restrict each cluster to its frequency). We see from this that `freq=52` is the most important cluster (because the loss increases a lot when this is removed), although clearly all clusters are important (because the loss is still very small if any one of them is ablated). We also see that ablating the always firing cluster has a very small effect, so clearly this cluster isn't very helpful for the task. (This is something we might have guessed beforehand, since the ReLU never firing makes this essentially a linear function, and in general having non-linearities allows you to learn much more expressive functions.)

In [ ]:
print("Loss with neuron activations excluding none:     {:.9f}".format(original_loss.item()))
for c, freq in enumerate(key_freqs_plus):
    print(
        "Loss with neuron activations excluding freq={}:  {:.9f}".format(
            freq,
            utils.test_logits(
                sum(logits_in_freqs) - logits_in_freqs[c],
                bias_correction=True,
                original_logits=original_logits,
            ),
        )
    )

## Understanding Logit Computation

> TLDR: The network uses $W_{logit}=W_{out}W_U$ to cancel out all 2D Fourier components other than the directions corresponding to $\cos(w(x+y)),\sin(w(x+y))$, and then multiplies these directions by $\cos(wz),\sin(wz)$ respectively and sums to get the output logits.

Recall that (for each neuron cluster with associated frequency $k$), each neuron's activations are a linear combination of const, linear and quadratic terms:

$$
\begin{bmatrix}
1 & \cos(\omega_k x) & \sin(\omega_k x) \\
\cos(\omega_k y) & \cos(\omega_k x)\cos(\omega_k y) & \sin(\omega_k x)\cos(\omega_k y) \\
\sin(\omega_k y) & \cos(\omega_k x)\sin(\omega_k y) & \sin(\omega_k x)\sin(\omega_k y)
\end{bmatrix}
$$

for $\omega_k = 2\pi k / p$.

To calculate the logits, the network cancels out all directions apart from:

$$
\begin{aligned}
\cos(\omega_k (x+y)) &= \cos(\omega_k x)\cos(\omega_k y)-\sin(\omega_k x)\sin(\omega_k y) \\
\sin(\omega_k (x+y)) &= \sin(\omega_k x)\cos(\omega_k y)+\cos(\omega_k x)\sin(\omega_k y)
\end{aligned}
$$

The network then multiplies these by $\cos(wz),\sin(wz)$ and sums (i.e. the logit for value $z$ will be the product of these with $\cos(wz),\sin(wz)$, summed over all neurons in that cluster, summed over all clusters).

#### Question - can you explain why this algorithm works?

<details>
<summary>Hint</summary>

Think about the following expression:

$$
\cos(\omega (x+y))\cos(\omega z)+\sin(\omega (x+y))\sin(\omega z)
$$

which is (up to a scale factor) the value added to the logit score for $z$.
</details>

<details>
<summary>Answer</summary>

The reason is again thanks to our trig formulas! Each neuron will add to the final logits a vector which looks like:

$$
\cos(w(x+y))\cos(wz)+\sin(w(x+y))\sin(wz)
$$

which we know from our trig formulas equals:

$$
\cos(w(x+y-z))
$$

which is largest when $z=x+y$.

---

Another way of writing this would be that, on inputs `(x, y)`, the model's logit output is (up to a scale factor) equal to:

$$
\cos(\omega(x+y-\vec{\textbf{z}})) = \begin{bmatrix}
\cos(\omega(x+y)) \\
\cos(\omega(x+y-1)) \\
\vdots \\
\cos(\omega(x+y-(p-1)))
\end{bmatrix}
$$

This vector is largest at element with index $x+y$, meaning the logit for $x+y$ will be largest (which is exactly what we want to happen, to solve our problem!).

Also, remember that we have several different frequencies $\omega_k$, and so when we sum over neurons, the vectors will combine constructively at $z = x+y$, and combine destructively everywhere else:

$$
f(t) = \sum_{k \in K} C_k \cos(\omega_k (x + y - \vec{\textbf{z}}))
$$

(where $C_k$ are large positive constants, which we'll explicitly calculate later on).
</details>

### Logits in Fourier Basis

To see that the network cancels out other directions, we can transform both the neuron activations and logits to the 2D Fourier Basis, and show the norm of the vector corresponding to each Fourier component - **we see that the quadratic terms have *much* higher norm in the logits than neuron activations, and linear terms are close to zero.** Remember that, to get from activations to logits, we apply the linear map $W_{logit}=W_{out}W_U$, which involves summing the outputs of all neurons.

Below is some code to visualise this. Note the use of `einops.reduce` rather than the `mean` method in the code below. Like most other uses of einops, this is useful because it's explicit and readable.

In [ ]:
utils.imshow_fourier(
    einops.reduce(neuron_acts_centered_fourier.pow(2), "y x neuron -> y x", "mean"),
    title="Norm of Fourier Components of Neuron Acts",
)

# Rearrange logits, so the first two dims represent (x, y) in modular arithmetic equation
original_logits_sq = einops.rearrange(original_logits, "(x y) z -> x y z", x=p)
original_logits_fourier = fft2d(original_logits_sq)
utils.imshow_fourier(
    einops.reduce(original_logits_fourier.pow(2), "y x z -> y x", "mean"),
    title="Norm of Fourier Components of Logits",
)

You should find that the linear and constant terms have more or less vanished relative to the quadratic terms, and that the quadratic terms are much larger in the logits than the neuron activations. This is annotated in the plots below (which should match the results you get from running the code):

### Exercise - validate by only taking quadratic terms

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> This exercise should feel similar to the previous one, since it's about vectors and projections.
> ```

Here, you will validate your results still further by *just* taking the components of the logits corresponding to $\cos(\omega_k(x+y))$ and $\sin(\omega_k(x+y))$ for each of our key frequencies $k$, and showing this increases performance.

First, you should write a function `get_trig_sum_directions`, which takes in a frequency `k` and returns the `(p, p)`-size vectors in 2D Fourier space corresponding to the directions:

$$
\begin{aligned}
\cos(\omega_k (\vec{\textbf{x}}+\vec{\textbf{y}})) &= \cos(\omega_k \vec{\textbf{x}})\cos(\omega_k \vec{\textbf{y}})-\sin(\omega_k \vec{\textbf{x}})\sin(\omega_k \vec{\textbf{y}}) \\
\sin(\omega_k (\vec{\textbf{x}}+\vec{\textbf{y}})) &= \sin(\omega_k \vec{\textbf{x}})\cos(\omega_k \vec{\textbf{y}})+\cos(\omega_k \vec{\textbf{x}})\sin(\omega_k \vec{\textbf{y}})
\end{aligned}
$$

respectively. Remember, the vectors you return should be normalized.

In [ ]:
def get_trig_sum_directions(k: int) -> tuple[Float[Tensor, "p p"], Float[Tensor, "p p"]]:
    """
    Given frequency k, returns the normalized vectors in the 2D Fourier basis representing the
    two directions:

        cos(ω_k * (x + y))
        sin(ω_k * (x + y))

    respectively.
    """
    cosx_cosy_direction = fourier_2d_basis_term(2 * k - 1, 2 * k - 1)
    sinx_siny_direction = fourier_2d_basis_term(2 * k, 2 * k)
    sinx_cosy_direction = fourier_2d_basis_term(2 * k, 2 * k - 1)
    cosx_siny_direction = fourier_2d_basis_term(2 * k - 1, 2 * k)

    cos_xplusy_direction = (cosx_cosy_direction - sinx_siny_direction) / np.sqrt(2)
    sin_xplusy_direction = (sinx_cosy_direction + cosx_siny_direction) / np.sqrt(2)

    return cos_xplusy_direction, sin_xplusy_direction


tests.test_get_trig_sum_directions(get_trig_sum_directions)

<details>
<summary>Hint</summary>

You can get the vector $\cos(\omega_k \vec{\textbf{x}}) \cos(\omega_k \vec{\textbf{y}})$ as follows:

```python
cosx_cosy_direction = fourier_2d_basis_term(2*k-1, 2*k-1)
```
</details>

Once you've passed these tests, you can run the code to project the logits onto these directions, and see how the loss changes. Note the use of the `project_onto_direction` function which you wrote earlier.

In [ ]:
trig_logits = []

for k in key_freqs:
    cos_xplusy_direction, sin_xplusy_direction = get_trig_sum_directions(k)
    cos_xplusy_projection = project_onto_direction(original_logits, cos_xplusy_direction.flatten())
    sin_xplusy_projection = project_onto_direction(original_logits, sin_xplusy_direction.flatten())

    trig_logits.extend([cos_xplusy_projection, sin_xplusy_projection])

trig_logits = sum(trig_logits)

print(f"Loss with just x+y components: {utils.test_logits(trig_logits, True, original_logits):.4e}")
print(f"Original Loss: {original_loss:.4e}")

You should find that the loss with just these components is significantly **lower** than your original loss. This is very strong evidence that we've correctly identified the algorithm used by our model.

### $W_{logit}$ in Fourier Basis

Okay, so we know that the model is mainly working with the terms $\cos(\omega_k(x+y))$ and $\sin(\omega_k(x+y))$ for each of our key frequencies $k$. Now, we want to show that the model's final ouput is:

$$
\cos(\omega_k (x + y - \vec{\textbf{z}})) = \cos(\omega_k (x + y))\cos(\omega_k \vec{\textbf{z}}) + \sin(\omega_k (x + y))\sin(\omega_k \vec{\textbf{z}})
$$

How do we do this?

Answer: ***we examine $W_{logit}$ in the Fourier basis.*** If we think that $W_{logit}$ is mainly projecting onto the directions $\cos(\omega_k \vec{\textbf{z}})$ and $\sin(\omega_k \vec{\textbf{z}})$, then we expect to find:

$$
W_{logit} \approx U S F = \sum_{i=1}^p \sigma_i u_i f_i^T
$$

where the singular values $\sigma_i$ are zero except those corresponding to Fourier basis vectors $f_i = \cos(\omega_k \vec{\textbf{z}}), \sin(\omega_k \vec{\textbf{z}})$ for key frequencies $k$. In other words:

$$
W_{logit} \approx \sum_{k \in K} \sigma_{2k-1} u_{2k-1} \cos(\omega_k \vec{\textbf{z}}) + \sigma_{2k} u_{2k} \sin(\omega_k \vec{\textbf{z}})
$$

Thus, if we right-multiply $W_{logit}$ by $F^T$, we should get a matrix $W_{logit} F^T \approx US$ of shape `(d_mlp, p)`, with all columns zero except for $\sigma_{2k-1}u_{2k-1}$ and $\sigma_{2k} u_{2k}$ for key frequencies $k$. Let's verify this:

In [ ]:
US = W_logit @ fourier_basis.T

utils.imshow_div(
    US, x=fourier_basis_names, yaxis="Neuron index", title="W_logit in the Fourier Basis"
)

You should see that the columns of this matrix are only non-zero at positions $2k$, $2k-1$ for the key frequencies. Since our model's final output is just a linear combination of these columns (with the coefficients given by the neuron activations), this proves that $W_{logit}$ is projecting onto directions corresponding to our key frequencies.

> Note the contrast between what we just did and what we've done before. In previous sections, we've taken the 2D Fourier transform of our activations / effective weights with respect to the input space (the vectors were $\vec{\textbf{x}}$ and $\vec{\textbf{y}}$). Now, we're taking the 1D Fourier tranformation with respect to the output space (the vectors are $\vec{\textbf{z}})$. It's pretty cool that this works!

So we've proven that:

$$
W_{logit} \approx \sum_{k \in K} \sigma_{2k-1} u_{2k-1} \cos(\omega_k \vec{\textbf{z}}) + \sigma_{2k} u_{2k} \sin(\omega_k \vec{\textbf{z}})
$$

but we still want to show that our final output is:

$$
f(t) = n_{post}^T W_{logit} \approx \sum_{k \in K} C_k \big(\cos(\omega_k (x + y)) \cos(\omega_k \vec{\textbf{z}}) + \sin(\omega_k (x + y)) \sin(\omega_k \vec{\textbf{z}})\big)
$$

where $n_{post} \in \mathbb{R}^{d_{mlp}}$ is the vector of neuron activations, and $C_k$ are large positive constants.

Matching coefficients of the vectors $\cos(\omega_k \vec{\textbf{z}})$ and $\sin(\omega_k \vec{\textbf{z}})$, this means we want to show that:

$$
\begin{aligned}
\sigma_{2k-1} u_{2k-1} &\approx C_k \cos(\omega_k (x + y)) \\
\sigma_{2k} u_{2k} &\approx C_k \sin(\omega_k (x + y))
\end{aligned}
$$

for each key frequency $k$.

First, let's do a quick sanity check. We expect vectors $u_{2k-1}$ and $u_{2k}$ to only contain components of frequency $k$, which means we expect the only non-zero elements of these vectors to correspond to the neurons in the $k$-frequency cluster. Let's test this by rearranging the matrix $W_{logit}F^T \approx US$, so that the neurons in each cluster are grouped together:

In [ ]:
US_sorted = t.concatenate([US[neuron_freqs == freq] for freq in key_freqs_plus])
hline_positions = np.cumsum(
    [(neuron_freqs == freq).sum().item() for freq in key_freqs]
).tolist() + [cfg.d_mlp]

utils.imshow_div(
    US_sorted,
    x=fourier_basis_names,
    yaxis="Neuron",
    title="W_logit in the Fourier Basis (rearranged by neuron cluster)",
    hline_positions=hline_positions,
    hline_labels=[f"Cluster: {freq=}" for freq in key_freqs.tolist()] + ["No freq"],
)

You should find that, for each frequency $k$, the components of the output in directions $\cos(\omega_k \vec{\textbf{z}})$ and $\sin(\omega_k \vec{\textbf{z}})$ are determined only by the neurons in the $k$-cluster, i.e. they are determined only by the 2D Fourier components of the input $(x, y)$ with frequency $k$.

This is promising, but we still haven't shown that $\sigma_{2k-1} u_{2k-1} \propto \cos(\omega_k(x+y))$, etc. To do this, we'll calculate the vectors $\sigma_{2k-1} u_{2k-1}$ and $\sigma_{2k} u_{2k}$ over all inputs $(x, y)$, then take the 2D Fourier transform.

In [ ]:
cos_components = []
sin_components = []

for k in key_freqs:
    sigma_u_sin = US[:, 2 * k]
    sigma_u_cos = US[:, 2 * k - 1]

    logits_in_cos_dir = neuron_acts_post_sq @ sigma_u_cos
    logits_in_sin_dir = neuron_acts_post_sq @ sigma_u_sin

    cos_components.append(fft2d(logits_in_cos_dir))
    sin_components.append(fft2d(logits_in_sin_dir))

for title, components in zip(["Cosine", "Sine"], [cos_components, sin_components]):
    utils.imshow_fourier(
        t.stack(components),
        title=f"{title} components of neuron activations in Fourier basis",
        animation_frame=0,
        animation_name="Frequency",
        animation_labels=key_freqs.tolist(),
    )

Can you interpret this plot? Can you explain why this plot confirms our hypothesis about how logits are computed?

<details>
<summary>Output (and explanation)</summary>

Recall we are trying to show that:

$$
\begin{aligned}
\sigma_{2k-1} u_{2k-1} &\approx C_k \cos(\omega_k (x + y)) \\
\sigma_{2k} u_{2k} &\approx C_k \sin(\omega_k (x + y))
\end{aligned}
$$

Writing this in the 2D Fourier basis, we get:

$$
\begin{aligned}
\sigma_{2k-1} u_{2k-1} &\approx \frac{C_k}{\sqrt{2}} \cos(\omega_k \vec{\textbf{x}})\cos(\omega_k \vec{\textbf{y}}) - \frac{C_k}{\sqrt{2}} \sin (\omega_k \vec{\textbf{x}})\sin (\omega_k \vec{\textbf{y}}) \\
\sigma_{2k} u_{2k} &\approx \frac{C_k}{\sqrt{2}} \cos(\omega_k \vec{\textbf{x}})\sin(\omega_k \vec{\textbf{y}}) + \frac{C_k}{\sqrt{2}} \sin (\omega_k \vec{\textbf{x}})\cos (\omega_k \vec{\textbf{y}})
\end{aligned}
$$

You should find that these exepcted 2D Fourier coefficients match the ones you get on your plot (i.e. they are approximately equal in magnitude, and the same sign for $\sin$ / opposite sign for $\cos$). For instance, zooming in on the $\cos$ plot for frequency $k=14$, we get:

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/cosine-components.png" width="650">
</details>

## Recap of section

Let's review what we've learned about each part of the network. We found that:

> #### Embedding
>
> The embedding projects onto a select few 1D Fourier basis vectors, corresponding to a handful of **key frequencies**.
>
> #### Neuron Activations
>
> Each neuron's activations are a linear combination of the constant term and the **linear and quadratic terms of a specific frequency**. The neurons clearly **cluster according to the key frequencies**.
>
> We found this by:
> * Seeing how well explained the neurons' variance was by a specific frequency, and finding that every neuron was well explained by a single frequency (except for a group of neurons which always fired, in other words they weren't doing anything nonlinear).
> * Projecting our neuron activations onto these corresponding frequencies, and showing that this didn't increase the overall loss by much.
>
> These activations are calculated by some nonlinear magic involving ReLUs and attention layers (because calculating things like the product of inputs at 2 different sequence positions is not a natural operation for a transformer to learn!).
>
> #### Logit Computation
>
> The network uses $W_{logit}=W_{out}W_U$ to cancel out all 2D Fourier components other than the directions corresponding to $\cos(w(x+y)),\sin(w(x+y))$, and then multiplies these directions by $\cos(wz),\sin(wz)$ respectively and sums to get the output logits.
>
> We found this by:
>
> * Showing that the linear terms more or less vanished from the logits, leaving only the quadratic terms behind
> * Each of our neurons seemed to be associated with a particular frequency, and the output of a neuron in frequency cluster $k$ directly affected the logits in the $k$-th Fourier basis.
> * The exact way that the neurons affected the logits at this frequency matched our initial guess of $\cos(\omega_k (x + y - \vec{\textbf{z}})) = \cos(\omega_k (x + y))\cos(\omega_k \vec{\textbf{z}}) + \sin(\omega_k (x + y))\sin(\omega_k \vec{\textbf{z}})$, i.e. each frequency cluster was responsible for computing this linear combination and adding it to the logit output.

# 3️⃣ Analysis During Training

> ##### Learning Objectives
>
> * Understand the idea of tracking metrics over time, and how this can inform when certain circuits are forming.
> * Investigate and interpret the evolution over time of the singular values of the model's weight matrices.
> * Investigate the formation of other capabilities in the model, like commutativity.

## Some starting notes

*Note - this section has fewer exercises than previous sections, and is intended more as a showcase of some of the results from the paper.*

In this section, we analyse the modular addition transformer during training. In the data we'll be using, checkpoints were taken every 100 epochs, from epoch 0 to 50K (the model we used in previous exercises was taken at 40K).

Usability note: I often use animations in this section. I recommend using the slider manually, not pressing play - Plotly smooths animations in a confusing and misleading way (and I haven't figured out how to fix it).

Usability note 2: To get plots to display, they can't have too many data points, so often plots will have different epoch intervals between data points, or different final epochs

Notation: I use "trig components" to refer to the components $\cos(\omega(x+y))$, $\sin(\omega(x+y))$.

## Overview

* The model starts to learn the generalised algorithm well before the phase change, and this develops fairly smoothly
    * We can see this with the metric of excluded loss, which seems to indicate that the model learns to "memorise more efficiently" by using the $\cos(w(x+y))$ directions.
    * This is a clear disproof of the 'grokking is a random walk in the loss landscape that eventually gets lucky' hypothesis.
* We also examine more qualitatively each circuit in the model and how it develops
    * We see that all circuits somewhat develop pre-grokking, but at different rates and some have a more pronounced phase change than others
    * We examine the embedding circuit, the 'calculating trig dimensions' circuit and the development of commutativity.
    * We also explore the development of neuron activations and how it varies by cluster
* There's a small but noticeable lag between 'the model learns the generalisable algorithm' and 'the model cleans up all memorised noise'.
* There are indications of several smaller phase changes, beyond the main grokking one.
    * In particular, a phase change at 43K-44K, well after grokking (I have not yet interpreted what's going on here).

## Setup

First, we'll define some useful functions. In particular, the `get_metrics` function is designed to populate a dictionary of metrics over the training period. The argument `metric_fn` is itself a function which takes in a model, and returns a metric (e.g. we use `metric_fn=test_loss`, to return the model's loss on the test set).

In [ ]:
# Define a dictionary to store our metrics in
metric_cache = {}


def get_metrics(model: HookedTransformer, metric_cache, metric_fn, name, reset=False):
    """
    Define a metric (by metric_fn) and add it to the cache, with the name `name`.

    If `reset` is True, then the metric will be recomputed, even if it is already in the cache.
    """
    if reset or (name not in metric_cache) or (len(metric_cache[name]) == 0):
        metric_cache[name] = []
        for sd in tqdm(full_run_data["state_dicts"]):
            model = utils.load_in_state_dict(model, sd)
            out = metric_fn(model)
            if isinstance(out, Tensor):
                out = to_numpy(out)
            metric_cache[name].append(out)
        model = utils.load_in_state_dict(model, full_run_data["state_dicts"][400])
        metric_cache[name] = t.tensor(np.array(metric_cache[name]))


def test_loss(model):
    logits = model(all_data)[:, -1, :-1]
    return utils.test_logits(logits, False, mode="test")


def train_loss(model):
    logits = model(all_data)[:, -1, :-1]
    return utils.test_logits(logits, False, mode="train")


epochs = full_run_data["epochs"]
plot_metric = partial(utils.lines, x=epochs, xaxis="Epoch")

get_metrics(model, metric_cache, test_loss, "test_loss")
get_metrics(model, metric_cache, train_loss, "train_loss")

## Excluded Loss

**Excluded Loss** for frequency $w$ is the loss on the training set where we delete the components of the logits corresponding to $\cos(w(x+y))$ and $sin(w(x+y))$. We get a separate metric for each $w$ in the key frequencies.

**Key observation:** The excluded loss (especially for frequency 14) starts to go up *well* before the point of grokking.

(Note: this performance decrease is way more than you'd get for deleting a random direction.)

### Exercise - implement excluded loss

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 20-30 minutes on this exercise.
> ```

You should fill in the function below to implement excluded loss. You'll need to use the `get_trig_sum_directions` and `project_onto_direction` functions from previous exercises. We've given you the first few lines of the function as a guide.

Note - when calculating the loss, you should use the `test_logits` function, with arguments `bias_correction=False, mode="train"`.

In [ ]:
def excl_loss(model: HookedTransformer, key_freqs: list) -> list:
    """
    Returns the excluded loss (i.e. subtracting the components of logits corresponding to
    cos(w_k(x+y)) and sin(w_k(x+y)), for each frequency k in key_freqs.
    """
    excl_loss_list = []
    logits = model(all_data)[:, -1, :-1]
    for freq in key_freqs:
        cos_xplusy_direction, sin_xplusy_direction = get_trig_sum_directions(freq)

        logits_cos_xplusy = project_onto_direction(logits, cos_xplusy_direction.flatten())
        logits_sin_xplusy = project_onto_direction(logits, sin_xplusy_direction.flatten())
        logits_excl = logits - logits_cos_xplusy - logits_sin_xplusy

        loss = utils.test_logits(logits_excl, bias_correction=False, mode="train").item()
        excl_loss_list.append(loss)

    return excl_loss_list


tests.test_excl_loss(excl_loss, model, key_freqs)

Once you've completed this function, you can run the following code to plot the excluded loss for each of the key frequencies (as well as the training and testing loss as a baseline). This plot should match the one at the end of the [Key Claims](https://www.lesswrong.com/posts/N6WM6hs7RQMKDhYjB/a-mechanistic-interpretability-analysis-of-grokking#Key_Claims) section of Neel Nanda's LessWrong post.

In [ ]:
get_metrics(model, metric_cache, partial(excl_loss, key_freqs=key_freqs), "excl_loss")

plot_metric(
    t.concat(
        [
            metric_cache["excl_loss"].T,
            metric_cache["train_loss"][None, :],
            metric_cache["test_loss"][None, :],
        ]
    ),
    labels=[f"excl {freq}" for freq in key_freqs] + ["train", "test"],
    title="Excluded Loss for each trig component",
    log_y=True,
    yaxis="Loss",
)

## Development of the embedding

### Embedding in Fourier basis

We can plot the norms of the embedding of each 1D Fourier component at each epoch. Pre-grokking, the model is learning the representation of prioritising a few components, but most components still have non-trivial value, presumably because these directions are doing some work in memorising. Then, during the grokking period, the other components get set to near zero - the model no longer needs other directions to memorise things, it's learned a general algorithm.

(As a reminder, we found that the SVD if the embedding was approximately $W_E \approx F^T S V^T$ where $F$ is the Fourier basis vector and $S$ is sparse, hence $F W_E \approx S V^T$ is also sparse with most rows equal to zero. So when we calculate the norm of the rows of $F W_E$ for intermediate points in training, we're seeing how the input space of the embedding learns to only contain these select few frequencies.)

### Exercise - define `fourier_embed`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> This exercise shouldn't take more than ~10 minutes.
> ```

Write the function `fourier_embed`. This should calculate norm of the Fourier transformation of the model's embedding matrix (ignoring the embedding vector corresponding to `=`). In other words, you should left-multiply the embedding matrix by the Fourier basis matrix, then calculate the sum of the norm of each embedding vector.

In [ ]:
def fourier_embed(model: HookedTransformer):
    """
    Returns norm of Fourier transform of the model's embedding matrix.
    """
    W_E_fourier = fourier_basis.T @ model.W_E[:-1]
    return einops.reduce(W_E_fourier.pow(2), "vocab d_model -> vocab", "sum")


tests.test_fourier_embed(fourier_embed, model)

Next, you can plot how the norm of Fourier components of the embedding changes during training:

In [ ]:
# Plot every 200 epochs so it's not overwhelming
get_metrics(model, metric_cache, fourier_embed, "fourier_embed")

utils.animate_lines(
    metric_cache["fourier_embed"][::2],
    snapshot_index=epochs[::2],
    snapshot="Epoch",
    hover=fourier_basis_names,
    animation_group="x",
    title="Norm of Fourier Components in the Embedding Over Training",
)

### Exercise - Examine the SVD of $W_E$

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

We discussed $W_E \approx F^T S V^T$ as being a good approximation to the SVD, but what happens when we actually calculate the SVD?

You should fill in the following function, which returns the singular values from the singular value decomposition of $W_E$ at an intermediate point in training. (Remember to remove the last row of $W_E$, which corresponds to the bias term.) PyTorch has an SVD function (`torch.svd`) which you should use for this.

In [ ]:
def embed_SVD(model: HookedTransformer) -> Tensor:
    """
    Returns vector S, where W_E = U @ diag(S) @ V.T in singular value decomp.
    """
    U, S, V = t.svd(model.W_E[:, :-1])
    return S


tests.test_embed_SVD(embed_SVD, model)

In [ ]:
get_metrics(model, metric_cache, embed_SVD, "embed_SVD")

utils.animate_lines(
    metric_cache["embed_SVD"],
    snapshot_index=epochs,
    snapshot="Epoch",
    title="Singular Values of the Embedding During Training",
    xaxis="Singular Number",
    yaxis="Singular Value",
)

Can you interpret what's going on in this plot?

<details>
<summary>Interpretation</summary>

At first, our SVD values are essentially random. Throughout training, the smaller singular values tend to zero, while the singular values corresponding to the key frequencies increase. Eventually, the graph demonstrates a sparse matrix, with all singular values zero except for those corresponding to the key frequencies.
</details>

***Note - after this point, there are no more exercises. The content just consists of plotting and interpreting/discussing results.***

## Development of computing trig components

In previous exercises, we've projected our logits or neuron activations onto 2D Fourier basis directions corresponding to $\cos(\omega_k(x+y))$ and $\sin(\omega_k(x+y))$ for each of the key frequencies $k$. We found that these directions explained basically all of the model's performance.

Here, we'll do the same thing, but over time, to see how the model learns to compute these trig components. The code is all provided for you below (it uses some functions you wrote in earlier sections).

The activations are first centered, then their sum of squares is taken, and then the Fourier components are extracted and we see what fraction of variance they explain. This is then averaged across the "output dimension" (which is neurons in the case of the neuron activations, or output classes in the case of the logits).

In [ ]:
def tensor_trig_ratio(model: HookedTransformer, mode: str):
    """
    Returns the fraction of variance of the (centered) activations which is explained by the Fourier
    directions corresponding to cos(ω(x+y)) and sin(ω(x+y)) for all the key frequencies.
    """
    logits, cache = model.run_with_cache(all_data)
    logits = logits[:, -1, :-1]
    if mode == "neuron_pre":
        tensor = cache["pre", 0][:, -1]
    elif mode == "neuron_post":
        tensor = cache["post", 0][:, -1]
    elif mode == "logit":
        tensor = logits
    else:
        raise ValueError(f"{mode} is not a valid mode")

    tensor_centered = tensor - einops.reduce(tensor, "xy index -> 1 index", "mean")
    tensor_var = einops.reduce(tensor_centered.pow(2), "xy index -> index", "sum")
    tensor_trig_vars = []

    for freq in key_freqs:
        cos_xplusy_direction, sin_xplusy_direction = get_trig_sum_directions(freq)
        cos_xplusy_projection_var = (
            project_onto_direction(tensor_centered, cos_xplusy_direction.flatten()).pow(2).sum(0)
        )
        sin_xplusy_projection_var = (
            project_onto_direction(tensor_centered, sin_xplusy_direction.flatten()).pow(2).sum(0)
        )
        tensor_trig_vars.extend([cos_xplusy_projection_var, sin_xplusy_projection_var])

    return to_numpy(sum(tensor_trig_vars) / tensor_var)


for mode in ["neuron_pre", "neuron_post", "logit"]:
    get_metrics(
        model,
        metric_cache,
        partial(tensor_trig_ratio, mode=mode),
        f"{mode}_trig_ratio",
        reset=True,
    )

lines_list = []
line_labels = []
for mode in ["neuron_pre", "neuron_post", "logit"]:
    tensor = metric_cache[f"{mode}_trig_ratio"]
    lines_list.append(einops.reduce(tensor, "epoch index -> epoch", "mean"))
    line_labels.append(f"{mode}_trig_frac")

plot_metric(
    lines_list,
    labels=line_labels,
    log_y=False,
    yaxis="Ratio",
    title="Fraction of logits and neurons explained by trig terms",
)

By plotting on a log scale, we can more clearly see that all 3 are having a higher proportion of trig components over training, but that the logits are smoother while the neurons exhibit more of a phase change.

### Discussion

In the fully trained model, there are two key components to the algorithm that results in the model being able to meaningfully use trig directions in the logits - firstly that the neuron activations have significant quadratic terms, and secondly that $W_{logit}$ can cancel out all of the non-trig terms, and then map the trig terms to `(x + y) % p`.

A natural question is whether one of these comes first, or if both evolve in tandem - as far as I'm aware, "how do circuits with multiple moving parts form over training" is not at all understood.

In this case, the logits develop the capability to cancel out everything but the trig directions early on, and the neurons don't develop significant quadratic or trig components until close to the grokking point.

I vaguely speculate that it makes more sense for circuits to develop in "reverse-order" - if we need two layers working together to produce a nice output, then if the second layer is randomly initialised the first layer can do nothing. But if the first layer is randomly initialised, the second layer can learn to just extract the components of the output corresponding to the "correct" output, and use them to badly approximate the output solution. And *now* the network has a training incentive to build up both parts of the circuit.

(This maybe has something to do with the Lottery Ticket hypothesis?)

## Development of neuron activations

There are two notable things about the neuron activations:
* They contain a significant component of quadratic terms of with x and y of the same frequency
* They group into clusters with Fourier terms of a single frequency

We can study the first one by plotting the fraction of a neuron's (centered) activation explained by the quadratic terms of that neuron's frequency (frequencies taken from the epoch 40K model)

(For the always firing cluster we sum over all frequencies).

In [ ]:
def get_frac_explained(model: HookedTransformer) -> Tensor:
    _, cache = model.run_with_cache(all_data, return_type=None)

    returns = []

    for neuron_type in ["pre", "post"]:
        neuron_acts = cache[neuron_type, 0][:, -1].clone().detach()
        neuron_acts_centered = neuron_acts - neuron_acts.mean(0)
        neuron_acts_fourier = fft2d(
            einops.rearrange(neuron_acts_centered, "(x y) neuron -> x y neuron", x=p)
        )

        # Calculate the sum of squares over all inputs, for each neuron
        square_of_all_terms = einops.reduce(
            neuron_acts_fourier.pow(2), "x y neuron -> neuron", "sum"
        )

        frac_explained = t.zeros(utils.d_mlp).to(device)
        frac_explained_quadratic_terms = t.zeros(utils.d_mlp).to(device)

        for freq in key_freqs_plus:
            # Get Fourier activations for neurons in this frequency cluster
            # We arrange by frequency (i.e. each freq has a 3x3 grid with const, linear & quadratic
            # terms)
            acts_fourier = arrange_by_2d_freqs(neuron_acts_fourier[..., neuron_freqs == freq])

            # Calculate the sum of squares over all inputs, after filtering for just this frequency
            # Also calculate the sum of squares for just the quadratic terms in this frequency
            if freq == -1:
                squares_for_this_freq = squares_for_this_freq_quadratic_terms = einops.reduce(
                    acts_fourier[:, 1:, 1:].pow(2), "freq x y neuron -> neuron", "sum"
                )
            else:
                squares_for_this_freq = einops.reduce(
                    acts_fourier[freq - 1].pow(2), "x y neuron -> neuron", "sum"
                )
                squares_for_this_freq_quadratic_terms = einops.reduce(
                    acts_fourier[freq - 1, 1:, 1:].pow(2), "x y neuron -> neuron", "sum"
                )

            frac_explained[neuron_freqs == freq] = (
                squares_for_this_freq / square_of_all_terms[neuron_freqs == freq]
            )
            frac_explained_quadratic_terms[neuron_freqs == freq] = (
                squares_for_this_freq_quadratic_terms / square_of_all_terms[neuron_freqs == freq]
            )

        returns.extend([frac_explained, frac_explained_quadratic_terms])

    frac_active = (neuron_acts > 0).float().mean(0)

    return t.nan_to_num(t.stack(returns + [neuron_freqs, frac_active], axis=0))


get_metrics(model, metric_cache, get_frac_explained, "get_frac_explained")

frac_explained_pre = metric_cache["get_frac_explained"][:, 0]
frac_explained_quadratic_pre = metric_cache["get_frac_explained"][:, 1]
frac_explained_post = metric_cache["get_frac_explained"][:, 2]
frac_explained_quadratic_post = metric_cache["get_frac_explained"][:, 3]
neuron_freqs_ = metric_cache["get_frac_explained"][:, 4]
frac_active = metric_cache["get_frac_explained"][:, 5]

utils.animate_scatter(
    t.stack([frac_explained_quadratic_pre, frac_explained_quadratic_post], dim=1)[:200:5],
    color=neuron_freqs_[:200:5],
    color_name="freq",
    snapshot="epoch",
    snapshot_index=epochs[:200:5],
    xaxis="Quad ratio pre",
    yaxis="Quad ratio post",
    title="Fraction of variance explained by quadratic terms (up to epoch 20K)",
)

utils.animate_scatter(
    t.stack([neuron_freqs_, frac_explained_pre, frac_explained_post], dim=1)[:200:5],
    color=frac_active[:200:5],
    color_name="frac_active",
    snapshot="epoch",
    snapshot_index=epochs[:200:5],
    xaxis="Freq",
    yaxis="Frac explained",
    hover=list(range(utils.d_mlp)),
    title="Fraction of variance explained by this frequency (up to epoch 20K)",
)

## Development of commutativity

We can plot the average attention to each position, and see that the model quickly learns to not attend to the final position, but doesn't really learn commutativity (ie equal attention to pos 0 and pos 1) until the grokking point.

**Aside:** There's a weird phase change at epoch 43K ish, where it starts to attend to position 2 again - I haven't investigated what's up with that yet.

(Each frame is 100 epochs)

In [ ]:
def avg_attn_pattern(model: HookedTransformer):
    _, cache = model.run_with_cache(all_data, return_type=None)
    return to_numpy(
        einops.reduce(cache["pattern", 0][:, :, 2], "batch head pos -> head pos", "mean")
    )


get_metrics(model, metric_cache, avg_attn_pattern, "avg_attn_pattern")

utils.imshow_div(
    metric_cache["avg_attn_pattern"][::5],
    animation_frame=0,
    animation_name="head",
    title="Avg attn by position and head, snapped every 100 epochs",
    xaxis="Pos",
    yaxis="Head",
    zmax=0.5,
    zmin=0.0,
    color_continuous_scale="Blues",
    text_auto=".3f",
)

We can also see this by plotting the average difference between pos 0 and pos 1.

In [ ]:
utils.lines(
    (metric_cache["avg_attn_pattern"][:, :, 0] - metric_cache["avg_attn_pattern"][:, :, 1]).T,
    labels=[f"Head {i}" for i in range(4)],
    x=epochs,
    xaxis="Epoch",
    yaxis="Average difference",
    title="Attention to pos 0 - pos 1 by head over training",
    width=900,
    height=450,
)

## Small lag to clean up noise

We plot test and train loss over training.

We further define **trig loss** as the loss where we extract out just the directions of the logits corresponding to $\cos(w(x+y)),\sin(w(x+y))$ in the key frequencies. We run this on all of the data, and on just the training set.

**Observations:**
* Trig loss on all data and train loss on just the training data are identical, showing that these dimensions are *only* used for a general algorithm treating train and test equally, rather than memorisation.
* Trig loss crashes before test loss crashes, and during the grokking period trig loss proportionately is much lower (by a factor of 10^4-10^5), but after grokking they return to a low ratio. This suggests that there's a small lag between the phase change where the model fully  learns the general algorithm and where it cleans up the noise left over by the memorisation circuit

**Aside:** Projecting onto the trig dimensions requires all of the data to be input into the model. To calculate the trig train loss, we first get the logits for *all* of the data, then project onto the trig components, then throw away the test data logits.

In [ ]:
def trig_loss(model: HookedTransformer, mode: str = "all"):
    logits = model(all_data)[:, -1, :-1]

    trig_logits = []
    for freq in key_freqs:
        cos_xplusy_dir, sin_xplusy_dir = get_trig_sum_directions(freq)
        cos_xplusy_proj = project_onto_direction(logits, cos_xplusy_dir.flatten())
        sin_xplusy_proj = project_onto_direction(logits, sin_xplusy_dir.flatten())
        trig_logits.extend([cos_xplusy_proj, sin_xplusy_proj])
    trig_logits = sum(trig_logits)

    return utils.test_logits(trig_logits, bias_correction=True, original_logits=logits, mode=mode)


get_metrics(model, metric_cache, trig_loss, "trig_loss")
get_metrics(model, metric_cache, partial(trig_loss, mode="train"), "trig_loss_train")

line_labels = ["test_loss", "train_loss", "trig_loss", "trig_loss_train"]
plot_metric(
    [metric_cache[lab] for lab in line_labels],
    labels=line_labels,
    title="Different losses over training",
)
plot_metric(
    [metric_cache["test_loss"] / metric_cache["trig_loss"]],
    title="Ratio of trig and test loss",
)

## Development of squared sum of the weights

Another data point is looking at the sum of squared weights for each parameter. Here we see several phases:

* (0-1K) The model first uses the neurons to memorise (which significantly increases the total weights of $W_{in}$ and $W_{out}$ but not the rest)
* (1K - 8K) It then smoothes out the computation across the model, so all weight matrices have the same total sum. In parallel, all matrices have total sum decreasing, presumably as it learns to use the trig directions.
* (8K-13K) It then groks the solution and things rapidly decrease. Presumably, it has learned how to use the trig directions well enough that it can clean up all other directions used for memorisation.
* (13K-43K) Then all weights plateau
* (43K-) In the total weight graph, we see a small but noticeable kink when we zoom in at this point, a final phase change. (955 to 942)

In [ ]:
def sum_sq_weights(model):
    return [param.pow(2).sum().item() for name, param in model.named_parameters()]


get_metrics(model, metric_cache, sum_sq_weights, "sum_sq_weights")

plot_metric(
    metric_cache["sum_sq_weights"].T,
    title="Sum of squared weights for each parameter",
    labels=[name.split(".")[-1] for name, _ in model.named_parameters()],
    log_y=False,
)
plot_metric(
    [einops.reduce(metric_cache["sum_sq_weights"], "epoch param -> epoch", "sum")],
    title="Total sum of squared weights",
    log_y=False,
)

# ☆ Bonus

## Discussion & Future Directions

The key claim I want to make is that grokking happens when the process of learning a general algorithm exhibits a phase change, *and* the model is given the minimal amount of data such that the phase change still happens.

### Why Phase Changes?

I further speculate that phase changes happen when a model learns *any* single generalising circuit that requires different parts of the model to work together in a non-linear way. Learning a sophisticated generalising circuit is hard and requires different parts of the model to line up correctly (ie coordinate on a shared representation). It doesn't seem super surprising if the circuit is in some sense either working or not, rather than smoothly increasing in performance

The natural next question is why the model learns the general algorithm at all - if performance is constant pre-phase change, there should be no gradient. Here I think the key is that it is hard to generalise to *unseen data*. The circuits pre-phase change are still fairly helpful for predicting seen data, as can be seen from [excluded loss](#scrollTo=Excluded_Loss). (though, also, test loss tends to decrease even pre-phase change, it's just slower - the question is of how large the gradient needs to be, not whether it's zero or not)

My current model is that every gradient update of the model is partially an incentive to memorise the data in the batch and partially to pick up patterns within the data. The gradient updates to memorise point in arbitrary directions and tend to cancel out (both within a batch and between batches) if the model doesn't have enough capacity to memorise fully, while the gradient updates to identify patterns reinforce each other. Regularisation is something that dampens the memorising gradients over the pattern recognition gradients. We don't observe this dynamic in the infinite data setting because we never re-run the model on past data points, but I expect it has at least slightly memorised previous training data. (See, eg, [Does GPT-2 Know Your Phone Number?](https://bair.berkeley.edu/blog/2020/12/20/lmmem/))

I speculate that this is because the model is incentivised to memorise the data as efficiently as possible, ie is biased towards simplicity. This comes both due to both explicit regularisation such as weight decay and implicit regularisation like inherent model capacity and SGD. The model learns to pick up patterns between data points, because these regularities allow it to memorise more efficiently. But this needs enough training data to make the memorisation circuit more complex than the generalising circuit(s).

This further requires there to be a few crisp circuits for the model to learn - if it fuzzily learns many different circuits, then lowering the amount of data will fairly smoothly decrease test performance, as it learns fewer and fewer circuits.


### Limitations

There are several limitations to this work and how confidently I can generalise from it to a general understanding of deep learning/grokking. In particular:
* The modular addition transformer is a toy model, which only needs a few circuits to solve the problem fully (unlike eg an LLM or an image model, which needs many circuits)
* The model is over-parametrised, as is shown by eg it learning many redundant neurons doing roughly the same task. It has many more parameters than it needs for near-perfect performance on its task, unlike real models.
* I only study weight decay as the form of regularisation
* My explanations of what's going on rely heavily on fairly fuzzy and intuitive notions that I have not explicitly defined: crisp circuits, simplicity, phase changes, what it means to memorise vs generalise, model capacity, etc.


### Relevance to Alignment

#### Model Training Dynamics

The main thing which seems relevant from an alignment point of view is a better understanding of model training dynamics.

In particular, phase changes seem pretty bad from an alignment perspective, because they suggest that rapid capability gain as models train or are scaled up is likely. This makes it less likely that we get warning shots (ie systems exhibiting non-sophisticated unaligned behaviour) before we get dangerous unaligned systems, and makes it less likely that systems near to AGI are good empirical test beds for alignment work. This work exhibits a bunch more examples of phase changes, and marginally updates me towards alignment being hard.

This work makes me slightly more hopeful, since it shows that phase changes are clearly foreshadowed by the model learning the circuits beforehand, and that we can predict capability gain with interpretability inspired metrics (though obviously this is only exhibited in a toy case, and only after I understood the circuit well - I'd rather not have to train and interpret an unaligned AGI before we can identify misalignment!).

More speculatively, it suggests that we may be able to use interpretability tools to shape training dynamics. A natural concern here is that a model may learn to obfuscate our tools, and perform the same algorithms but without them showing up on our tools. This could happen both from gradient descent directly learning to obfuscate things (because the unaligned solution performs much better than an aligned solution), or because the system itself has [learned to be deceptive and alter itself to avoid our tools](https://www.lesswrong.com/posts/nbq2bWLcYmSGup9aF/a-transparency-and-interpretability-tech-tree). The fact that we observe the circuits developing well before they are able to generalise suggests that we might be able to disincentivise deception before the model gets good enough at deception to be able to generalise and evade deveption detectors (though doesn't do much for evading gradient descent).

The natural future direction here is to explore training on interpretability inspired metrics, and to see how much gradient descent learns to Goodhart them vs shifting its inductive bias to learn a different algorithm. Eg, can we incentive generalisation and get grokking with less data? Can we incentivise memorisation and change the algorithm it learns? What happens if we disincentivise learning to add with certain frequencies?

#### Other Relevance

This also seems relevant as more evidence of the [circuits hypothesis](https://distill.pub/2020/circuits/zoom-in/), that networks *can* be mechanistically understood and are learning interpretable algorithms. And also as one of the first examples of interpreting a transformer's neurons (though on a wildly different task to language).

It also seems relevant as furthering the science of deep learning by better understanding how models generalise, and the weird phenomena of grokking. Understanding whether a model will be aligned or not requires us to understand how it will generalise, and to predict future model behaviour, and what different training setups will and will not generalise. This motivation is somewhat more diffuse and less directed, but seems analogous to how statistical learning theory allows us to predict things about models in classical statistics (though is clearly insufficient for deep learning).

(Though honestly I mostly worked on this because it was fun, I was on holiday, and I got pretty nerd-sniped by the problem. So all of this is somewhat ad-hoc and backwards looking, rather than this being purely alignment motivated work)


### Training Dynamics

One interesting thing this research gives us insight into is the training dynamics of circuits - which parts of the circuit develop first, at what rates, and why?

My speculative guess is that, when a circuit has several components interacting between layers, the component closest to the logits is easier to form and will tend to form first.

**Intuition:** Imagine a circuit involving two components interacting with a non-linearity in the middle, which are both randomly initialised. They want to converge on a shared representation, involving a few directions in the hidden space. Initially, the random first component will likely have some output corresponding to the correct features, but with small coefficients and among a lot of noise. The second component can learn to focus on these correct features and cancel out the noise, reinforcing the incentive for the first component to focus on these features. On the other hand, if the second component is random, it's difficult to see how the first component can produce reasonable features that are productively used by the second component.

Qualitatively, we observe that all circuits are forming in parallel pre-grokking, but it roughly seems that the order is logit circuit > embedding circuit > neuron circuit > attention circuit (ie logit is fastest, attention is slowest)

This seems like an interesting direction of future research by giving a concrete example of crisp circuits that are easy to study during training. Possible initial experiments would be fixing parts of the network to be random, initialising parts of the network to be fully trained and frozen, or giving different parts of the network different learning rates.

## Suggested capstone projects

### Investigate phase changes

You could look for other examples of phase changes, for example:

* Toy problems
    * Something incentivising skip trigrams
    * Something incentivising virtual attention heads
    * e.g. one of the models below (or pick an easier task)
* Looking for [curve detectors](https://distill.pub/2020/circuits/curve-circuits) in a ConvNet
    * A dumb way to try this would be to train a model to imitate the actual curve detectors in Inception (eg minimising OLS loss between the model's output and curve detector activations)
* Looking at the formation of interpretable neurons in a [SoLU transformer](https://transformer-circuits.pub/2022/solu/index.html)
* Looking inside a LLM with many checkpoints
    * Eleuther have many checkpoints of GPT-J and GPT-Neo, and will share if you ask
    * [Mistral](https://nlp.stanford.edu/mistral/getting_started/download.html) have public versions of GPT-2 small and medium, with 5 runs and many checkpoints
    * Possible capabilities to look for
        * Performance on benchmarks, or specific questions from benchmarks
        * Simple algorithmic tasks like addition, or sorting words into alphabetical order, or matching open and close brackets
        * Soft induction heads, eg [translation](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html#performing-translation)
        * Look at attention heads on various text and see if any have recognisable attention patterns (eg start of word, adjective describing current word, syntactic features of code like indents or variable definitions, most recent open bracket, etc).

### Try more algorithmic problems

Interpreting toy models is a good way to increase your confidence working with TransformerLens and basic interpretability methods. It's maybe not the most exciting category of open problems in mechanistic interpretability, but can still be a useful exercise - and sometimes it can lead to interesting new insights about how interpretability tools can be used.

If you're feeling like it, you can try to hop onto LeetCode and pick a suitable problem (we recommend the "Easy" section) to train a transformer and interpret its output. Here are a few suggestions to get you started (some of these were taken from LeetCode, others from Neel Nanda's [open problems post](https://www.lesswrong.com/s/yivyHaCAmMJ3CqSyj/p/ejtFsvyhRkMofKAFy)). They're listed approximately from easier to harder, although this is just a guess since I haven't personally interpreted these. Note, there are ways you could make any of these problems easier or harder with modifications - I've included some ideas inline.

* Calculating sequences with a Fibonacci-style recurrence relation (i.e. predicting the next element from the previous two)
* [Search Insert Position](https://leetcode.com/problems/search-insert-position/) - an easier version would be when the target is always guaranteed to be in the list (you also wouldn't need to worry about sorting in this case). The version without this guarantee is a very different problem, and would be much harder
* [Is Subsequence](https://leetcode.com/problems/is-subsequence/) - you should start with subsequences of length 1 (in which case this problem is pretty similar to the easier version of the previous problem), and work up from there
* [Majority Element](https://leetcode.com/problems/majority-element/) - you can try playing around with the data generation process to change the difficulty, e.g. sequences where there is no guarantee on the frequency of the majority element (i.e. you're just looking for the token which appears more than any other token) would be much harder
* [Number of Equivalent Domino Pairs](https://leetcode.com/problems/number-of-equivalent-domino-pairs/) - you could restrict this problem to very short lists of dominos to make it easier (e.g. start with just 2 dominos!)
* [Longest Substring Without Repeating Characters](https://leetcode.com/problems/longest-substring-without-repeating-characters/)
* [Isomorphic Strings](https://leetcode.com/problems/isomorphic-strings/) - you could make it simpler by only allowing the first string to have duplicate characters, or by restricting the string length / vocabulary size
* [Plus One](https://leetcode.com/problems/plus-one/) - you might want to look at the "sum of numbers" algorithmic problem before trying this, and/or the grokking exercises in this chapter. Understanding this problem well might actually help you build up to interpreting the "sum of numbers" problem (I haven't done this, so it's very possible you could come up with a better interpretation of that monthly problem than [mine](https://www.perfectlynormal.co.uk/blog-november-monthly-problem), since I didn't go super deep into the carrying mechanism)
* Predicting permutations, i.e. predicting the last 3 tokens of the 12-token sequence `(17 3 11) (17 1 13) (11 2 4) (11 4 2)` (i.e. the model has to learn what permutation function is being applied to the first group to get the second group, and then apply that permutation to the third group to correctly predict the fourth group). Note, this problem might require 3 layers to solve - can you see why?
* Train models for [automata](https://arxiv.org/pdf/2210.10749.pdf) tasks and interpret them - do your results match the theory?
* Predicting the output to simple code functions. E.g. predicting the `1 2 4` text in the following sequence (which could obviously be made harder with some obvious modifications, e.g. adding more variable definitions so the model has to attend back to the right one):
```python
a = 1 2 3
a[2] = 4
a -> 1 2 4
```

* Graph theory problems like [this](https://jacobbrazeal.wordpress.com/2022/09/23/gpt-3-can-find-paths-up-to-7-nodes-long-in-random-graphs/). You might have to get creative with the input format when training transformers on tasks like this!

Note, ARENA runs a [monthly algorithmic problems sequence](https://arena-ch1-transformers.streamlit.app/Monthly_Algorithmic_Problems), and you can get ideas from looking at past problems from this sequence. You can also use these repos to get some sample code for building & training a trnasformer on a toy model, and constructing a dataset for your particular problem.

## Suggested paper replications

### [A Toy Model of Universality: Reverse Engineering How Networks Learn Group Operations](https://arxiv.org/abs/2302.03025)

This paper extends the analysis of this particular task & model, by looking at a general group operation of which modular addition is a special case.

This might be a good replication for you if:

* You enjoyed all subsections in this exercise set, and would like to perform similar analysis on more complex algorithms
* You're interested in studying grokking and training dynamics
* You have a background in mathematics, and in particular have some familiarity with group theory (and ideally some representation theory)